In [ ]:
#@title Ячейка 0 - импорт doc_map (с авто-монтированием Drive)
import sys, os, importlib

# 1. монтируем Drive, если ещё не примонтирован
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. добавляем путь к проекту
BASE = "/content/drive/MyDrive/rag_exp"
if BASE not in sys.path:
    sys.path.insert(0, BASE)

# 3. сбрасываем кэш импортов (лечит "ModuleNotFoundError" после монтирования)
importlib.invalidate_caches()

# 4. импорт
from doc_map import FILE_TO_ID
print("Загружено документов:", len(FILE_TO_ID))


Mounted at /content/drive
Загружено документов: 11


In [ ]:
#@title Ячейка 1 · Фаза 1 — боевой прогон + метрики §3.1
# Старт: ячейки 2-11 (deps → Drive → GPU → модель → реранкер → OpenAI → import → init_common).
# Делает: читает 11 PDF (7 старых + 4 новых: gost_32569_2013, nd_2_020101_127_ch5,
#         ost_5r_4110, ost_5_6066_75)
#         → build_or_load_index (корпус изменился → ПЕРЕСБОРКА, новый key) → SparseIndex → run_grid на QA_REAL.
# Результат: phase1_grid.csv. Связь с rag_01 — через index_cache на Drive.


In [ ]:
#@title Ячейка 2 — установка зависимостей
!pip install -q -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" bitsandbytes accelerate
!pip install -q scikit-learn pandas pyyaml rank_bm25 openai pymupdf
print("deps ok")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 109.3 MB/s eta 0:00:00
deps ok


In [ ]:
#@title Ячейка 3 — Drive, BASE, HF_HOME, секреты
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/rag_exp"
os.makedirs(f"{BASE}/results", exist_ok=True)
os.environ.pop("HF_HOME", None)
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY"); print("OPENAI_API_KEY: ok")
except Exception:
    print("OPENAI_API_KEY: НЕ задан (S8/S9 не сработают)")
print("BASE =", BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPENAI_API_KEY: ok
BASE = /content/drive/MyDrive/rag_exp


In [ ]:
#@title Ячейка 4 — проверка GPU
import torch
assert torch.cuda.is_available(), "GPU не подключён! Среда выполнения → Сменить → T4 GPU"
print(torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


NVIDIA L4 | VRAM 23.7 GB


In [ ]:
#@title Ячейка 5 — загрузка модели Qwen3-4B-int8
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen3-Embedding-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
bnb = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModel.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print(f"VRAM после загрузки: {torch.cuda.memory_allocated()/1e9:.1f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

VRAM после загрузки: 4.4 GB


In [ ]:
#@title Ячейка 6 — реранкер (для S2/S5/S7/S9)
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512,
                        model_kwargs={"torch_dtype": torch.float16}, device="cuda")
print(f"VRAM с реранкером: {torch.cuda.memory_allocated()/1e9:.1f} GB")


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

VRAM с реранкером: 5.6 GB


In [ ]:
#@title Ячейка 7 — OpenAI клиент (для S8/S9)
oai = None
if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        oai = OpenAI()
        # лёгкая проверка, что клиент реально живой
        print("OpenAI ok | ключ длиной:", len(os.environ["OPENAI_API_KEY"]), "символов")
    except Exception as e:
        oai = None
        print("OpenAI клиент НЕ создан:", type(e).__name__, "— S8/S9 будут пропущены")
else:
    print("OPENAI_API_KEY не найден в окружении — S8/S9 недоступны (для Фазы 1 не нужны)")

OpenAI ok | ключ длиной: 164 символов


In [ ]:
#@title Ячейка 8 — импорт rag_common + init_common
import sys, importlib
sys.path.append(BASE)
importlib.invalidate_caches()
import rag_common; importlib.reload(rag_common)
from rag_common import init_common, build_or_load_index, SparseIndex, run_grid, compare_strategies

init_common(BASE, MODEL, model, tokenizer, reranker, oai)
rag_common.model = model
rag_common.tokenizer = tokenizer
rag_common.MODEL = MODEL

print("rag_common ok | tokenizer:", type(rag_common.tokenizer).__name__,
      "| reranker:", type(reranker).__name__,
      "| oai:", "есть" if oai else "нет")

rag_common init: MODEL=Qwen/Qwen3-Embedding-4B, INDEX_CACHE=/content/drive/MyDrive/rag_exp/index_cache, reranker=ok, oai=ok
rag_common ok | tokenizer: Qwen2Tokenizer | reranker: CrossEncoder | oai: есть


In [ ]:
#@title Ячейка 8.5 — патч encode (батчевый)
import rag_common, torch, time
import torch.nn.functional as F
# init_common уже вызван в Ячейке 8 (с reranker и oai) — здесь НЕ повторяем,
# иначе можно сбросить reranker/oai. Тут только патчим encode.

@torch.no_grad()
def encode_batched(texts, dim=None, max_length=512, batch_size=16):
    out = []
    total = len(texts)
    t0 = time.time()
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        enc = rag_common.tokenizer(batch, padding=True, truncation=True,
                                   max_length=max_length, return_tensors="pt").to(rag_common.model.device)
        hidden = rag_common.model(**enc).last_hidden_state
        vecs = rag_common.last_token_pool(hidden, enc["attention_mask"])
        if dim:
            vecs = vecs[:, :dim]
        vecs = F.normalize(vecs, p=2, dim=1)
        out.append(vecs.float().cpu())
        del enc, hidden, vecs
        torch.cuda.empty_cache()
        if (i // batch_size) % 10 == 0:
            print(f"  эмбеддинг: {i+len(batch)}/{total}  ({time.time()-t0:.0f} сек)")
    print(f"  готово: {total} за {time.time()-t0:.0f} сек")
    return torch.cat(out, dim=0)

rag_common.encode = encode_batched
print("encode:", rag_common.encode.__name__,
      "| model:", type(rag_common.model).__name__,
      "| tokenizer:", type(rag_common.tokenizer).__name__)

encode: encode_batched | model: Qwen3Model | tokenizer: Qwen2Tokenizer


In [ ]:
#@title Ячейка 9 — чтение всех PDF корпуса → real_docs
import fitz, os
CORPUS_DIR = f"{BASE}/corpus_phase1"
# FILE_TO_ID уже загружен сверху (из doc_map / автосборки) — НЕ переопределяем!
real_docs = []
for fname, doc_id in FILE_TO_ID.items():
    pdf = fitz.open(os.path.join(CORPUS_DIR, fname))
    real_docs.append({"id": doc_id, "text": "\n".join(p.get_text() for p in pdf)})
    pdf.close()
    print(f"{doc_id:22s} <- {fname}")
print("Документов:", len(real_docs))



pkps_chast_11          <- ПКПС Часть XI (Электрическое оборудование, изд.2017).pdf
pkps_chast_8           <- ПКПС Часть VIII (Системы и трубопроводы, изд.2018).pdf
rd_50_726_92           <- РД 50-726-92.pdf
gost_2_701             <- ГОСТ 2.701-2008.pdf
gost_30893_1           <- ГОСТ 30893.1-2002.pdf
gost_32569_2013        <- ГОСТ 32569-2013 (Трубопроводы технологические стальные).pdf
nd_2_020101_127_ch5    <- НД 2-020101-127 ч.5 (Правила ПОМС, навигационное оборудование).pdf
nd_2_09_006_kn6        <- НД 2-09-006 кн.6 (переиздан как 2-039901-005, 2018).pdf
nd_2_020101_174_ch2    <- НД 2-020101-174 ч.2 (Правила РС, корпус).pdf
ost_5r_4110            <- ОСТ 5Р.4110-2003 (Монтаж механизмов).pdf
ost_5_6066_75          <- ОСТ 5.6066-75 (Электромонтаж, заземление).pdf
Документов: 11


In [ ]:
#@title Ячейка 9.1 - проверка длины извлечённого текста по каждому документу
print(f"{'doc_id':24s} {'символов':>10s}")
print("-" * 36)
for d in real_docs:
    n = len(d["text"])
    flag = "  ⚠️ ПУСТО/СКАН" if n < 1000 else ""
    print(f"{d['id']:24s} {n:>10,d}{flag}")
print("-" * 36)
print(f"Всего документов: {len(real_docs)}")


doc_id                     символов
------------------------------------
pkps_chast_11               517,249
pkps_chast_8                462,765
rd_50_726_92                130,509
gost_2_701                   41,739
gost_30893_1                 17,713
gost_32569_2013             334,494
nd_2_020101_127_ch5         683,132
nd_2_09_006_kn6             375,828
nd_2_020101_174_ch2         785,091
ost_5r_4110                  89,657
ost_5_6066_75                 6,185
------------------------------------
Всего документов: 11


In [ ]:
#@title Ячейка 10 — индекс (корпус 10 PDF → ПЕРЕСБОРКА, не HIT) + SparseIndex
idx_real = build_or_load_index(real_docs, dim=2048, quant="int8")
sparse_real = SparseIndex(idx_real)
print(f"Чанков: {len(idx_real['chunks'])}, key={idx_real['key']}")


[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive
Чанков: 7493, key=0724bbc94fa070d9


In [ ]:
#@title Ячейка 11 — QA_REAL из реестра (боевой набор)
import pandas as pd
df_reg = pd.read_csv(f"{BASE}/qa_registry.csv")
ready = df_reg[df_reg["статус"].isin(["GOOD","GOOD_OTHER_ED","GOOD_SYNTHETIC"])].copy()

# имя колонки с текстом вопроса: подстрахуемся
qcol = "вопрос_полный" if "вопрос_полный" in df_reg.columns else "вопрос"

QA_REAL = []
for _, row in ready.iterrows():
    QA_REAL.append({
        "query": str(row[qcol]),
        "relevant": {row["doc_id"]: 3},
        "Q": int(row["Q"]),
        "категория": row["категория"],
    })
print(f"QA_REAL: {len(QA_REAL)} вопросов")
print("По категориям:", ready["категория"].value_counts().to_dict())
print("По документам:", ready["doc_id"].value_counts().to_dict())
print("Пустые doc_id:", ready["doc_id"].isna().sum() + (ready["doc_id"]=="").sum())

QA_REAL: 44 вопросов
По категориям: {'A': 17, 'C': 9, 'B': 9, 'D': 5, 'E': 4}
По документам: {'pkps_chast_8': 14, 'nd_2_09_006_kn6': 8, 'nd_2_020101_174_ch2': 8, 'pkps_chast_11': 3, 'rd_50_726_92': 2, 'ost_5_6066_75': 2, 'gost_2_701': 2, 'gost_30893_1': 2, 'nd_2_020101_127_ch5': 1, 'gost_32569_2013': 1, 'ost_5r_4110': 1}
Пустые doc_id: 0


In [ ]:
#@title Ячейка 11.1 · Чанковая разметка QA_REAL (Путь A: 1 эталонный чанк = rel 3)
# После: ячейка 11 (старый QA_REAL документного уровня).
# Делает: переопределяет QA_REAL → {chunk_id: 3}. Эталон: из примечания (номер чанка),
#         иначе по номеру пункта (первый чанк-заголовок, глубина ≥ X.Y.Z), иначе fallback {doc_id: 3}.
#         Сравнительные вопросы ('чем отличается/различается', 'vs') → принудительно документный уровень.
# Источник: qa_registry.csv + idx_real. Результат: чанковый QA_REAL + сводка покрытия.
# Перед: ячейка 12 (run_grid). ВНИМАНИЕ: запускать строго после 11, до 12.
import re, pandas as pd

df = pd.read_csv(f"{BASE}/qa_registry.csv")
good = df[df["статус"].isin(["GOOD","GOOD_SYNTHETIC","GOOD_OTHER_ED"])].copy()
chunks = idx_real["chunks"]; meta = idx_real["chunk_meta"]

def norm(s): return re.sub(r"\s+", " ", str(s).lower()).strip()
nchunks = [norm(c) for c in chunks]
by_doc = {}
for i, m in enumerate(meta):
    by_doc.setdefault(m["doc_id"], []).append(i)

def chunk_from_note(note):
    """первый номер чанка из примечания вида 'эталонные чанки: 4668, 4670'"""
    if pd.isna(note): return None
    m = re.search(r"чанк[а-я]*\s*:?\s*(\d+)", str(note), re.I)
    return int(m.group(1)) if m else None

def clause_from_note(note):
    """пункт вида 'п.18.2.1.2' из примечания, если колонка 'пункт' пуста"""
    if pd.isna(note): return None
    m = re.search(r"п\.?\s*(\d+(?:\.\d+)+)", str(note))
    return m.group(1) if m else None

def first_clause(clause):
    """первый составной номер X.Y(.Z...) из 'A vs B', 'A-B' и т.п."""
    if pd.isna(clause): return None
    nums = re.findall(r"\d+(?:\.\d+)+", str(clause))
    return nums[0] if nums else None

def is_comparative(clause, question):
    """сравнительные вопросы требуют двух эталонов → не годятся для Пути A"""
    s = f"{clause} {question}".lower()
    return (" vs " in str(clause).lower()) or ("отлич" in s) or ("различ" in s)

def chunk_from_clause(doc_id, clause):
    if clause is None or doc_id not in by_doc: return None
    pat = re.escape(clause.strip()).replace(r"\.", r"\.\s?")
    rx = re.compile(pat)
    for i in by_doc[doc_id]:
        if rx.search(nchunks[i]): return i
    return None

QA_REAL = []
cov_note, cov_clause, cov_doc = [], [], []
for _, row in good.iterrows():
    qid, doc_id = row.get("Q","?"), row["doc_id"]
    q = {"id": qid, "question": row["вопрос"], "doc_id": doc_id}
    gold, src = chunk_from_note(row.get("примечание")), None

    if gold is not None:
        src = "note"; cov_note.append(qid)
    elif is_comparative(row.get("пункт"), row["вопрос"]):
        gold = None                                  # сравнительные → документный
    else:
        clause = first_clause(row.get("пункт")) or clause_from_note(row.get("примечание"))
        if clause and clause.count(".") >= 2:        # глубина ≥ X.Y.Z, режем '1.1'/'разд.1'
            gold = chunk_from_clause(doc_id, clause)
            if gold is not None: src = "clause"; cov_clause.append(qid)

    if gold is not None:
        q["relevant"], q["level"], q["gold_source"] = {gold: 3}, "chunk", src
    else:
        q["relevant"], q["level"] = {doc_id: 3}, "doc"; cov_doc.append(qid)
    QA_REAL.append(q)

# --- сводка покрытия ---
print(f"Всего вопросов: {len(QA_REAL)}")
print(f"  чанк из примечания: {len(cov_note)} → {cov_note}")
print(f"  чанк по пункту:     {len(cov_clause)} → {cov_clause}")
print(f"  документный:        {len(cov_doc)} → {cov_doc}")
print(f"\nИтого на уровне чанков: {len(cov_note)+len(cov_clause)} из {len(QA_REAL)}")
print("\n--- остались документными ---")
for q in QA_REAL:
    if q["level"] == "doc":
        print(f"  Q{q['id']:>3} | {q['doc_id']:22s} | {q['question'][:55]}")


Всего вопросов: 44
  чанк из примечания: 5 → [2, 9, 10, 22, 31]
  чанк по пункту:     25 → [3, 8, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 24, 25, 27, 28, 29, 33, 34, 35, 37, 41, 43, 44, 45]
  документный:        14 → [1, 5, 6, 21, 23, 30, 36, 38, 39, 40, 42, 46, 47, 48]

Итого на уровне чанков: 30 из 44

--- остались документными ---
  Q  1 | rd_50_726_92           | Когда нужен экранированный кабель
  Q  5 | gost_2_701             | Разница схем соединений и подключения
  Q  6 | nd_2_09_006_kn6        | Размеры вырезов в балках
  Q 21 | pkps_chast_8           | Класс трубопровода и группа очистки
  Q 23 | pkps_chast_8           | Смотровые окна на цистернах
  Q 30 | gost_30893_1           | Общие допуски размеров на чертеже
  Q 36 | pkps_chast_8           | Чем отличаются требования к воздушным трубам от требова
  Q 38 | nd_2_020101_174_ch2    | Чем отличается требование к минимальной толщине палубы 
  Q 39 | nd_2_09_006_kn6        | Чем различаются требования к вырезам в балках и к п

In [ ]:
#@title Ячейка 12 — чанковый грид (S1-S7) → phase2_chunk_grid.csv
# ПРИМЕЧАНИЕ: здесь chunk_size=512 — ранний прогон. Итоговое решение (chunk_size=1024)
# принято в rag_03 по результатам Фазы 2A. Подробности — в report_phase2.md.
PARAM_GRID = {"chunk_strategy":["fixed_512"], "chunk_size":[512],
              "overlap":[0.1], "dim":[2048], "quant":["int8"]}
PHASE1_CSV = f"{BASE}/results/phase2_chunk_grid.csv"   # новый файл, не затираем phase1
SEVEN = ["S1","S2","S3","S4","S5","S6","S7"]
df_real = run_grid(real_docs, QA_REAL, PARAM_GRID,
                   results_csv=PHASE1_CSV, sparse_index=sparse_real,
                   strategies=SEVEN, append=False)
print(df_real[["strategy","P@5","R@5","MRR","NDCG@5","Hit@5","MAP","search_sec"]].to_string(index=False))

Конфигураций: 1 × стратегий: 7 = 7 строк → phase2_chunk_grid.csv

[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive
[1/1] cfg={'chunk_strategy': 'fixed_512', 'chunk_size': 512, 'overlap': 0.1, 'dim': 2048, 'quant': 'int8'}  индекс готов за 0.4s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  эмбеддинг: 1/1  (1 сек)
  готово: 1 за 1 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 

In [ ]:
#@title Ячейка 12.1 — чанковый грид (S8-S9, append) → phase2_chunk_grid.csv
PARAM_GRID = {"chunk_strategy":["fixed_512"], "chunk_size":[512],
              "overlap":[0.1], "dim":[2048], "quant":["int8"]}
PHASE2_CSV = f"{BASE}/results/phase2_chunk_grid.csv"
TWO = ["S8","S9"]
df_89 = run_grid(real_docs, QA_REAL, PARAM_GRID,
                 results_csv=PHASE2_CSV, sparse_index=sparse_real,
                 strategies=TWO, append=True)
print(df_89[["strategy","P@5","R@5","MRR","NDCG@5","Hit@5","MAP","search_sec"]].to_string(index=False))


Конфигураций: 1 × стратегий: 2 = 2 строк → phase2_chunk_grid.csv

[cache HIT] индекс 0724bbc94fa070d9: 7493 чанков загружено с Drive
[1/1] cfg={'chunk_strategy': 'fixed_512', 'chunk_size': 512, 'overlap': 0.1, 'dim': 2048, 'quant': 'int8'}  индекс готов за 0.4s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 1/1  (0 сек)
  готово: 1 за 0 сек
  эмбеддинг: 

In [ ]:
#@title Ячейка 12.2 — дедупликация phase2_chunk_grid.csv + итоговая таблица
import pandas as pd
PHASE2_CSV = f"{BASE}/results/phase2_chunk_grid.csv"

df = pd.read_csv(PHASE2_CSV)
before = len(df)
df = df.drop_duplicates(subset=["chunk_size","overlap","dim","strategy"], keep="last")
if len(df) < before:
    df.to_csv(PHASE2_CSV, index=False)
    print(f"Удалено дублей: {before-len(df)} → строк: {len(df)}")
else:
    print(f"Дублей нет, строк: {len(df)} (ожидаем 9)")

# итоговая таблица, отсортированная по NDCG@5
print("\n=== Чанковый грид, 9 стратегий (сортировка по NDCG@5) ===")
cols = ["strategy","P@5","R@5","MRR","NDCG@5","Hit@5","MAP","search_sec"]
print(df.sort_values("NDCG@5", ascending=False)[cols].to_string(index=False))

Дублей нет, строк: 9 (ожидаем 9)

=== Чанковый грид, 9 стратегий (сортировка по NDCG@5) ===
strategy    P@5    R@5    MRR  NDCG@5  Hit@5    MAP  search_sec
      S5 0.3364 1.6818 0.5415  1.0660 0.6591 2.4140       17.21
      S7 0.3227 1.6136 0.5294  1.0281 0.6591 2.4085       16.98
      S2 0.3273 1.6364 0.5143  1.0274 0.6591 2.4330       16.28
      S1 0.3091 1.5455 0.4584  0.9476 0.6136 2.3122       13.95
      S6 0.2864 1.4318 0.4181  0.8636 0.5682 2.0903       13.87
      S4 0.2773 1.3864 0.4053  0.8274 0.5455 2.1058       13.60
      S3 0.2318 1.1591 0.3153  0.6778 0.4773 1.5983        0.75
      S8 0.0636 0.3182 0.2955  0.3014 0.3182 0.2955      371.28
      S9 0.0636 0.3182 0.2311  0.2530 0.3182 0.2311      440.17
